# Smart Campus IoT · Crowd Management

Run the code cell below in local Jupyter Notebook or JupyterLab. Set up the live walkthrough or use the **Scenario** menu and **Next step** for the scripted walkthrough.

1. Choose a **Radar room**. To customise that facility's starting count, enter a **Start count** and click **Set start & apply**. This saves the value for **Reset Demo** until you restart the kernel. Repeat for other facilities as needed.
2. Close Arduino Serial Monitor, enter your ESP32 port (default `COM5`), then click **Connect ESP32**. Use your current Arduino sketch that prints lines such as `TARGET 1  ENTER | Campus Gym | People in room: 1/20 | LOW`.
3. Walk through the doorway. Each ENTER or EXIT updates the facility selected in Jupyter. The Arduino's printed room name and cumulative occupancy are ignored. Finish your crossing before switching facilities.
4. Click **Reset Demo** between audiences. It restores all saved starting counts, the queue, demo clock and initial booking statuses. The serial connection and selected facility stay active.

**Live demo defaults:** Food Court A and B start at 46/60, Library Common Study Space at 30/40, Common Study B at 22/30, and Campus Gym at 18/20. The first row assigned to radar is Food Court A. The five count-based facilities can receive the radar. Discussion rooms use their booking scenarios.

**Scripted demos:** Disconnect ESP32, select a scenario, and use **Start scenario** and **Next step**. Scenarios use scripted values where needed. Food-court waits and bookings are simulated. The gym queue can be demonstrated with its scripted scenario.

**Setup:** If needed, run `%pip install ipywidgets pyserial` in a separate cell. `pyserial` is only used for live radar. **Export Sheets CSV** creates a snapshot you can import into Google Sheets.


In [2]:
import asyncio
import re
from datetime import datetime
from getpass import getpass
from html import escape

import ipywidgets as w
import requests
from IPython.display import display

LIB = "Library Common Study Space"
A, B = "Food Court A", "Food Court B"
STUDY, GYM = "W5 Common Study Space B", "Campus Gym"

CAPACITY = {LIB: 40, A: 60, B: 60, STUDY: 30, GYM: 20}
ROOMS = [f"Discussion Room {i}" for i in range(1, 4)]
DEMO_ROOM = ROOMS[2]
FACILITY_OPTIONS = [*CAPACITY, *ROOMS]
BOOKING_MINUTES = 15
SECONDS_PER_DEMO_MINUTE = 1.5
SCENARIOS = [
    "Food Court A reaches HIGH",
    "Both food courts are HIGH",
    "Library fills up",
    "Gym goes from LOW to HIGH",
    "Discussion rooms and booking release",
    "Crowd reduction",
]

if "serial_task" in globals() and serial_task is not None:
    serial_task.cancel()
serial_task = None
old_booking_task = globals().get("booking_task")
if old_booking_task is not None:
    old_booking_task.cancel()
booking_task = None
booking_minutes_left = None
booking_expired = False
booking_checked_in = False

seed_counts = {LIB: 30, A: 46, B: 46, STUDY: 22, GYM: 18}
room_seeds = dict(zip(ROOMS, (0, 1, 0)))
radar_rooms = set()
counts, previous = {}, {}
waits = {}
bookings = {}
arduino_event = re.compile(r"^TARGET [1-3]\s+(ENTER|EXIT)\s*\|")

def change(room, count):
    if counts[room] != count:
        previous[room], counts[room] = counts[room], count

def crowd_status(room):
    n, cap = counts[room], CAPACITY[room]

    if room != GYM:
        if n == 0:
            return "NONE"
        if n > cap:
            return "OVER CAPACITY"
    if n * 100 >= cap * 80:
        return "HIGH"
    return "MEDIUM" if n * 100 >= cap * 40 else "LOW"

def room_status(room):
    booked, present = bookings[room]

    if booked:
        if room == DEMO_ROOM and booking_checked_in:
            return "BOOKED"
        return "BOOKED & OCCUPIED" if present else "BOOKED BUT VACANT"
    return "OCCUPIED" if present else "AVAILABLE"

def snapshot():
    rows = []

    for room, cap in CAPACITY.items():
        n = counts[room]
        detail = []

        if room != GYM and cap * 70 <= n * 100 < cap * 80 and n > previous[room]:
            detail.append("APPROACHING HIGH")
        if room == LIB and n > previous[room]:
            detail.append("Rising trend · may get busier")
        if room in waits:
            detail.append(f"{waits[room]} min estimated seating wait (demo)")

        source = (
            "Demo + radar crossings" if room in radar_rooms
            else "Simulated occupancy"
        )
        rows.append([room, f"{n}/{cap}", crowd_status(room), " · ".join(detail), source])

    for room in ROOMS:
        booked, present = bookings[room]
        detail = ""
        if room == DEMO_ROOM and booking_minutes_left is not None:
            detail = f"{booking_minutes_left} min to release"
        elif room == DEMO_ROOM and booking_checked_in:
            detail = "Student checked in"
        source = "Demo booking + radar crossings" if room in radar_rooms else "Simulated booking"
        rows.append([room, str(present), room_status(room), detail, source])

    return rows

HEADERS = ["Facility", "Occupancy", "Status", "Details", "Source"]
table = w.HTML()
recommendation = w.HTML()
step_note = w.HTML()
serial_note = w.HTML("Live radar disconnected")

def refresh():
    colors = {
        "HIGH": "#9e2327",
        "MEDIUM": "#875200",
        "LOW": "#12614a",
        "AVAILABLE": "#12614a",
        "BOOKED BUT VACANT": "#875200",
    }

    header = "".join(
        f'<th style="padding:10px;text-align:left;background:#8e1616;color:white">{h}</th>'
        for h in HEADERS
    )

    body = []
    for i, row in enumerate(snapshot()):
        cells = []

        for j, value in enumerate(row):
            color = colors.get(value, "#202027") if j == 2 else "#202027"
            weight = "700" if j in (0, 2) else "400"
            cells.append(
                f'<td style="padding:9px;color:{color};font-weight:{weight}">'
                f"{escape(value)}</td>"
            )

        background = "#f7f4f4" if i % 2 == 0 else "#ffffff"
        body.append(
            f'<tr style="background:{background}">' + "".join(cells) + "</tr>"
        )

    table.value = (
        '<table style="border-collapse:collapse;width:100%">'
        f"<thead><tr>{header}</tr></thead><tbody>"
        + "".join(body)
        + "</tbody></table>"
    )

    suggestions = []
    high_food_courts = [
        room for room in (A, B)
        if counts[room] * 5 >= 4 * CAPACITY[room]
    ]

    if len(high_food_courts) == 1:
        alternative = B if high_food_courts[0] == A else A
        suggestions.append(
            f"Recommend {alternative} · less crowded "
            f"({counts[alternative]}/{CAPACITY[alternative]})."
        )
    elif len(high_food_courts) == 2 and A in waits and B in waits:
        if waits[A] == waits[B]:
            suggestions.append(
                "Both food courts are crowded and have equal estimated waits."
            )
        else:
            best = min((A, B), key=waits.get)
            suggestions.append(
                f"Recommend {best} · shorter demo wait ({waits[best]} min)."
            )

    if counts[LIB] * 5 >= 4 * CAPACITY[LIB] and counts[STUDY] * 5 < 2 * CAPACITY[STUDY]:
        suggestions.append(f"Recommend {STUDY} for studying.")

    recommendation.value = "<br>".join(escape(s) for s in suggestions)

def stop_booking_countdown():
    global booking_task, booking_minutes_left, booking_expired

    if booking_task is not None:
        booking_task.cancel()
    booking_task = None
    booking_minutes_left = None
    booking_expired = False

def reset():
    global counts, previous, waits, bookings, booking_checked_in

    stop_booking_countdown()
    booking_checked_in = False

    counts = seed_counts.copy()
    previous = counts.copy()
    radar_rooms.clear()
    waits = {A: 9, B: 3}
    bookings = {room: (room == ROOMS[1], room_seeds[room]) for room in ROOMS}

def apply_step(name, step):
    if name == SCENARIOS[0]:
        change(B, 18)
        change(A, [41, 45, 48][step])
        waits[A] = [9, 11, 14][step]
        return [
            "Food Court A starts at MEDIUM.",
            "Food Court A is approaching HIGH.",
            "A reaches 80%. Compare the two demo waits.",
        ][step]

    if name == SCENARIOS[1]:
        change(A, 48)
        change(B, [47, 48, 48][step])
        waits.update({A: [14, 14, 8][step], B: [3, 10, 10][step]})
        return [
            "Food Court A is HIGH. Food Court B needs one more entry to reach HIGH.",
            "Both courts are HIGH. B has the shorter wait.",
            "The demo waits change. A now has the shorter wait.",
        ][step]

    if name == SCENARIOS[2]:
        change(STUDY, 9)
        change(LIB, [24, 29, 36][step])
        return [
            "Library starts at MEDIUM.",
            "Library occupancy is rising through 70–79%.",
            "Library reaches HIGH. W5 Common Study Space B is the alternative.",
        ][step]

    if name == SCENARIOS[3]:
        change(GYM, [7, 8, 16][step])
        return [
            "Gym starts at 7/20 · LOW.",
            "Gym reaches 8/20 · MEDIUM.",
            "Gym reaches 16/20 · HIGH.",
        ][step]

    change(A, [48, 47, 22][step])
    return [
        "Food Court A starts at HIGH.",
        "Exits reduce A to MEDIUM.",
        "More exits reduce A to LOW.",
    ][step]

scenario = w.Dropdown(
    options=SCENARIOS, description="Scenario", layout=w.Layout(width="440px")
)
start = w.Button(description="Start scenario", button_style="primary")
next_step = w.Button(description="Next step", disabled=True)
reset_demo_button = w.Button(description="Reset Demo", button_style="warning")
radar_picker = w.Dropdown(
    options=FACILITY_OPTIONS, value=A, description="Facility Type",
    style={"description_width": "initial"},
    layout=w.Layout(width="430px"),
)
preset = w.BoundedIntText(
    value=seed_counts[A], min=0, max=CAPACITY[A], description="Start count"
)
save_preset = w.Button(
    description="Set start & apply", layout=w.Layout(width="155px")
)
preset_note = w.HTML()
room_picker = w.Dropdown(
    options=FACILITY_OPTIONS, description="Simulate for",
    style={"description_width": "initial"}, layout=w.Layout(width="430px"),
)
plus = w.Button(description="Simulate ENTER +1")
minus = w.Button(description="Simulate EXIT −1")
book_room = w.Button(description="Book Room 3", button_style="primary")
booking_note = w.HTML()
port = w.Text(value="COM5", description="Serial port")
connect = w.Button(description="Connect ESP32")
disconnect = w.Button(description="Disconnect", disabled=True)
current_step = None

def play(step):
    global current_step

    if scenario.value == SCENARIOS[4]:
        current_step = None
        next_step.disabled = True
        step_note.value = "Use Book Room 3 below to start the automatic countdown."
        return

    if serial_task is not None:
        step_note.value = "Disconnect the ESP32 before starting a scripted scenario."
        return

    if step == 0:
        reset()

    current_step = step
    step_note.value = escape(apply_step(scenario.value, step))
    next_step.disabled = step == 2
    refresh()

def manual(delta):
    room = room_picker.value

    if serial_task is not None and room == radar_picker.value:
        step_note.value = "Use doorway crossings to update the active radar facility."
        return

    change_facility_occupancy(room, max(0, facility_occupancy(room) + delta))
    refresh()

async def run_booking_countdown():
    global booking_task, booking_minutes_left, booking_expired

    task = asyncio.current_task()
    loop = asyncio.get_running_loop()
    started = loop.time()

    try:
        for elapsed in range(1, BOOKING_MINUTES + 1):
            await asyncio.sleep(
                max(0, started + elapsed * SECONDS_PER_DEMO_MINUTE - loop.time())
            )
            booking_minutes_left = BOOKING_MINUTES - elapsed

            if booking_minutes_left == 0:
                bookings[DEMO_ROOM] = (False, 0)
                booking_minutes_left = None
                booking_note.value = "Room 3 timer reached 0. The room is now available."
                booking_expired = True

            refresh()
    finally:
        if booking_task is task:
            booking_task = None

def book_discussion_room(_):
    global booking_task, booking_minutes_left, booking_expired

    booked, people = bookings[DEMO_ROOM]
    if booked or people:
        booking_note.value = "Room 3 is already booked or occupied."
        return

    loop = asyncio.get_running_loop()
    radar_picker.value = DEMO_ROOM
    bookings[DEMO_ROOM] = (True, 0)
    booking_minutes_left = BOOKING_MINUTES
    booking_expired = False
    booking_task = loop.create_task(run_booking_countdown())
    booking_note.value = "Room 3 booked. Each demo minute takes 1.5 seconds."
    refresh()

def reset_demo():
    global current_step

    reset()
    current_step = None
    next_step.disabled = True
    step_note.value = "Demo reset to saved starting counts. Bookings reset."
    booking_note.value = ""
    refresh()

def select_radar_room():
    room = radar_picker.value
    starting_count = seed_counts[room] if room in CAPACITY else room_seeds[room]
    preset.max = CAPACITY.get(room, 2147483647)
    preset.value = starting_count
    preset_note.value = f"Sensor events update <b>{escape(room)}</b>."

    if serial_task is not None:
        serial_note.value = f"Listening for the next crossing for <b>{escape(room)}</b>."

def apply_preset():
    room = radar_picker.value

    if room in CAPACITY:
        seed_counts[room] = preset.value
        counts[room] = previous[room] = preset.value
    else:
        room_seeds[room] = preset.value
        change_facility_occupancy(room, preset.value)

    radar_rooms.discard(room)
    preset_note.value = f"Saved and applied {preset.value} for <b>{escape(room)}</b>."
    refresh()

def facility_occupancy(room):
    return counts[room] if room in CAPACITY else bookings[room][1]

def change_facility_occupancy(room, people):
    global booking_checked_in

    if room in CAPACITY:
        change(room, people)
        return

    booked, _ = bookings[room]
    bookings[room] = (booked, people)
    if room == DEMO_ROOM and people > 0 and booking_minutes_left is not None:
        booking_checked_in = True
        stop_booking_countdown()
        booking_note.value = "Room 3 checked in. The booking countdown stopped."

def apply_radar_event(line):
    event = arduino_event.match(line)
    if not event:
        return None

    room = radar_picker.value
    direction = event.group(1)
    delta = 1 if direction == "ENTER" else -1
    change_facility_occupancy(room, max(0, facility_occupancy(room) + delta))
    radar_rooms.add(room)
    return room, direction

async def read_radar(connection):
    global serial_task

    pending = bytearray()
    last_message = "Live radar disconnected."

    try:
        while True:
            pending.extend(connection.read(connection.in_waiting))
            lines = pending.split(b"\n")
            pending = bytearray(lines.pop()[-4096:])

            for raw in lines:
                line = raw.decode("utf-8", errors="replace").strip()
                crossing = apply_radar_event(line)

                if crossing is not None:
                    room, direction = crossing
                    serial_note.value = (
                        f"{direction} · {escape(room)} · {escape(connection.port)}"
                    )
                    refresh()

            await asyncio.sleep(0.10)

    except asyncio.CancelledError:
        last_message = "Live radar disconnected. Facility counts are retained."
        raise
    except Exception as exc:
        last_message = f"Serial connection ended · {exc}"
    finally:
        connection.close()
        serial_task = None
        scenario.disabled = start.disabled = connect.disabled = False
        disconnect.disabled = True
        next_step.disabled = current_step is None or current_step >= 2
        serial_note.value = escape(last_message)
        refresh()

def open_serial(_):
    global serial_task

    if serial_task is not None:
        return

    try:
        import serial

        loop = asyncio.get_running_loop()
        connection = serial.Serial(port.value.strip(), 115200, timeout=0)
    except ImportError:
        serial_note.value = "Install pyserial in this notebook kernel with %pip install pyserial."
        return
    except Exception as exc:
        serial_note.value = (
            f"Could not open the port. Close Arduino Serial Monitor. {escape(str(exc))}"
        )
        return

    connect.disabled = start.disabled = scenario.disabled = next_step.disabled = True
    disconnect.disabled = False
    serial_note.value = (
        f"Listening on {escape(connection.port)} for crossings "
        f"for <b>{escape(radar_picker.value)}</b>."
    )
    serial_task = loop.create_task(read_radar(connection))

def close_serial(_):
    if serial_task is not None:
        serial_task.cancel()

WEB_APP_URL = (
    "https://script.google.com/macros/s/AKfycbyZ2CdIEQCj-FLNCXgc64FoZ8oS0Hr-c6hdr61V_Urdil3Cbt0kE2TGo5ovckaFy6z9/exec"
)

if not globals().get("api_key"):
    api_key = getpass("Crowd Management API key (Enter to skip publishing): ").strip()

facility_ids = {
    A: "FCA",
    B: "FCB",
    LIB: "LIB",
    STUDY: "STB",
    GYM: "GYM",
    ROOMS[0]: "DR1",
    ROOMS[1]: "DR2",
    ROOMS[2]: "DR3",
}

def facility_suggestion(room):
    if room in (A, B):
        other = B if room == A else A
        room_high = counts[room] * 5 >= CAPACITY[room] * 4
        other_high = counts[other] * 5 >= CAPACITY[other] * 4

        if room_high and not other_high:
            return facility_ids[other]
        if room_high and other_high and room in waits and other in waits:
            if waits[other] < waits[room]:
                return facility_ids[other]

    if room == LIB:
        library_high = counts[LIB] * 5 >= CAPACITY[LIB] * 4
        study_low = counts[STUDY] * 5 < CAPACITY[STUDY] * 2
        if library_high and study_low:
            return facility_ids[STUDY]

    return ""

def build_sheet_row(room, timestamp):
    row = {
        "Timestamp": timestamp,
        "Facility ID": facility_ids[room],
        "Facility Name": room,
        "Data Source": next(item[4] for item in snapshot() if item[0] == room),
    }

    if room in CAPACITY:
        count = counts[room]
        capacity = CAPACITY[room]

        if count > capacity:
            raise ValueError(
                f"{room} shows {count}/{capacity}. "
                "The existing Sheet script rejects counts above capacity."
            )

        approaching = (
            room != GYM
            and 70 * capacity <= 100 * count < 80 * capacity
            and count > previous[room]
        )

        row.update({
            "Current Occupancy": count,
            "Max Occupancy": capacity,
            "Occupancy (%)": round(100 * count / capacity, 1),
            "Crowd Status": crowd_status(room),
            "Approaching HIGH": "YES" if approaching else "NO",
            "Estimated Wait (min)": waits.get(room, ""),
            "Facility Suggestion": facility_suggestion(room),
        })
    else:
        row["Booking Status"] = room_status(room)
        row["Current Occupancy"] = facility_occupancy(room)
        if room == DEMO_ROOM:
            if booking_minutes_left is not None:
                row["Time Room Has Been Empty (min)"] = BOOKING_MINUTES - booking_minutes_left
            elif booking_expired and room_status(room) == "AVAILABLE":
                row["Time Room Has Been Empty (min)"] = BOOKING_MINUTES
            else:
                row["Time Room Has Been Empty (min)"] = ""

    return row

sheet_note = w.HTML()
sheet_send_lock = globals().get("sheet_send_lock") or asyncio.Lock()

def post_sheet_rows(rows):
    response = requests.post(
        WEB_APP_URL, json={"api_key": api_key, "rows": rows}, timeout=30,
    )
    response.raise_for_status()
    return response.json()

async def sync_all_to_sheet():
    if not api_key:
        sheet_note.value = "Sheets sync is off. Rerun the dashboard cell and enter your Crowd Management API key."
        return

    last_saved = {}
    while True:
        retry_delay = 1
        try:
            async with sheet_send_lock:
                timestamp = datetime.now().astimezone().isoformat(timespec="seconds")
                rows = []
                pending = {}
                for room in FACILITY_OPTIONS:
                    row = build_sheet_row(room, timestamp)
                    signature = {key: value for key, value in row.items() if key != "Timestamp"}
                    if last_saved.get(room) != signature:
                        rows.append(row)
                        pending[room] = signature

                if rows:
                    sheet_note.value = f"⏳ Sending <b>{len(rows)} facility update(s)</b> to Google Sheets…"
                    request = asyncio.create_task(asyncio.to_thread(post_sheet_rows, rows))
                    try:
                        result = await asyncio.shield(request)
                    except asyncio.CancelledError:
                        await asyncio.gather(request, return_exceptions=True)
                        raise

                    if not isinstance(result, dict) or result.get("success") is not True or result.get("updated") != len(rows):
                        message = result.get("error") if isinstance(result, dict) else None
                        raise ValueError(message or "Google Sheets did not confirm all facility updates.")

                    last_saved.update(pending)
                    sent_at = datetime.now().astimezone().strftime("%H:%M:%S")
                    sheet_note.value = f"✅ Sent <b>{len(rows)} facility update(s)</b> to Google Sheets at {sent_at}."

        except requests.exceptions.RequestException as exc:
            retry_delay = 5
            sheet_note.value = f"⚠️ Sheets sync failed ({type(exc).__name__}). Retrying…"
        except ValueError as exc:
            retry_delay = 5
            message = escape(str(exc).replace(api_key, "[redacted]"))
            sheet_note.value = f"⚠️ Sheets sync failed: {message} Retrying…"

        await asyncio.sleep(retry_delay)

previous_study_tasks = globals().get("study_tasks", ())
for task in previous_study_tasks:
    task.cancel()
study_tasks = ()
study_link = w.ToggleButton(
    description="Enable Energy link", layout=w.Layout(width="190px"),
)
mqtt_note = w.HTML("MQTT link off.")

mqtt_send_lock = globals().get("mqtt_send_lock") or asyncio.Lock()

ENERGY_BROKER = "broker.emqx.io"
ENERGY_PORT = 1883
ENERGY_CROWD_TOPIC = "sitg7/room1/crowd"
ENERGY_LEVELS = {"NONE": 0, "LOW": 1, "MEDIUM": 2, "HIGH": 3, "OVER CAPACITY": 3}

def send_study_mqtt(level):
    from paho.mqtt import publish

    publish.single(
        ENERGY_CROWD_TOPIC, str(level), qos=1, retain=False,
        hostname=ENERGY_BROKER, port=ENERGY_PORT, keepalive=10,
    )

async def sync_study_mqtt():
    loop = asyncio.get_running_loop()
    last_level = None
    next_heartbeat = 0
    while True:
        band = crowd_status(STUDY)
        level = ENERGY_LEVELS[band]
        if level != last_level or loop.time() >= next_heartbeat:
            try:
                async with mqtt_send_lock:
                    publish_task = asyncio.create_task(asyncio.to_thread(send_study_mqtt, level))
                    try:
                        await asyncio.shield(publish_task)
                    except asyncio.CancelledError:
                        await asyncio.gather(publish_task, return_exceptions=True)
                        raise
            except Exception as exc:
                mqtt_note.value = f"MQTT publish failed ({type(exc).__name__}). Retrying…"
                await asyncio.sleep(3)
                continue
            last_level = level
            next_heartbeat = loop.time() + 5
            mqtt_note.value = f"W5 Common Study Space B → MQTT: <b>{band} ({level})</b>"
        await asyncio.sleep(0.2)

def toggle_study_link(change):
    global study_tasks
    if change["new"]:
        try:
            import paho.mqtt
        except ImportError:
            mqtt_note.value = "Run %pip install paho-mqtt in a notebook cell, then enable the link."
            study_link.value = False
            return
        study_tasks = (
            asyncio.create_task(sync_study_mqtt()),
        )
        study_link.description = "Disable Energy link"
    else:
        for task in study_tasks:
            task.cancel()
        study_tasks = ()
        study_link.description = "Enable Energy link"
        if "paho-mqtt" not in mqtt_note.value:
            mqtt_note.value = "MQTT link off."

study_link.observe(toggle_study_link, names="value")

start.on_click(lambda _: play(0))
next_step.on_click(lambda _: play(current_step + 1))
scenario.observe(lambda _: play(0), names="value")
radar_picker.observe(lambda _: select_radar_room(), names="value")
save_preset.on_click(lambda _: apply_preset())
reset_demo_button.on_click(lambda _: reset_demo())
plus.on_click(lambda _: manual(1))
minus.on_click(lambda _: manual(-1))
book_room.on_click(book_discussion_room)
connect.on_click(open_serial)
disconnect.on_click(close_serial)

display(w.VBox([
    w.HTML('<h2 style="color:#8e1616;margin:0">Smart Campus · Crowd Management</h2>'),
    w.HBox(
        [scenario, start, next_step, reset_demo_button],
        layout=w.Layout(flex_flow="row wrap"),
    ),
    step_note,
    table,
    w.VBox([study_link, sheet_note, mqtt_note], layout=w.Layout(margin="8px 0 12px 0")),
    w.VBox([recommendation], layout=w.Layout(margin="0 0 18px 0")),
    w.VBox([
        w.HBox([radar_picker, preset, save_preset], layout=w.Layout(flex_flow="row wrap")),
        preset_note,
    ], layout=w.Layout(margin="0 0 20px 0")),
    w.HBox(
        [room_picker, plus, minus],
        layout=w.Layout(flex_flow="row wrap", margin="0 0 10px 0"),
    ),
    w.HBox(
        [port, connect, disconnect],
        layout=w.Layout(flex_flow="row wrap"),
    ),
    serial_note,
    book_room,
    booking_note,
]))

reset_demo()
select_radar_room()

previous_sheet_task = globals().get("auto_sheet_task")
if previous_sheet_task is not None:
    previous_sheet_task.cancel()
sheet_note.value = "Preparing automatic Google Sheets sync…"
auto_sheet_task = asyncio.get_running_loop().create_task(sync_all_to_sheet())


In [2]:
from getpass import getpass

api_key = getpass("Paste the current CROWD_API_KEY: ").strip()
print("Key loaded." if api_key else "No key entered.")


Paste the current CROWD_API_KEY:  ········


Key loaded.


In [3]:
%pip install paho-mqtt


Note: you may need to restart the kernel to use updated packages.
